# Vanilla GCN for Semi-Supervised Node Classification

This notebook implements a two-layer Vanilla Graph Convolutional Network (GCN)
for semi-supervised node classification.

The implementation follows the mathematical propagation rule:

$$
H^{(l+1)}
=
\sigma
\left(
\hat{A}H^{(l)}W^{(l)}
\right)
$$

where the normalized adjacency matrix is:

$$
\hat{A}
=
\tilde{D}^{-\frac{1}{2}}
\tilde{A}
\tilde{D}^{-\frac{1}{2}}
$$

The implementation is performed directly using matrix multiplication so that
each computational step corresponds to the mathematical GCN formulation.

The graph contains 5 nodes and 2 classes. Nodes 0 and 3 are used as labeled
training nodes, while the remaining nodes are treated as unlabeled nodes.

The complete workflow is:

$$
A
\rightarrow
\tilde{A}
\rightarrow
\tilde{D}
\rightarrow
\tilde{D}^{-\frac{1}{2}}
\rightarrow
\hat{A}
\rightarrow
X
\rightarrow
H^{(1)}
\rightarrow
Z
\rightarrow
P
\rightarrow
\mathcal{L}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Training}
$$

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Display settings
np.set_printoptions(precision=6, suppress=True)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

ModuleNotFoundError: No module named 'torch'

## 1. Define the Graph

We consider an undirected graph:

$$
G=(V,E)
$$

with five nodes:

$$
V=\{0,1,2,3,4\}
$$

and the following undirected edges:

$$
E=
\{
(0,1),
(1,2),
(2,3),
(3,4),
(4,0),
(0,2)
\}
$$

The graph consists of a 5-node cycle with an additional edge between
nodes 0 and 2.

Since the graph is undirected, every edge contributes to both directions
in the adjacency matrix.

In [ ]:
# Number of nodes
num_nodes = 5

# Undirected edges
edges = [
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 0),
    (0, 2)
]

print("Number of nodes:", num_nodes)
print("Edges:")
for edge in edges:
    print(edge)

## 2. Construct the Adjacency Matrix

The adjacency matrix $A$ represents the connectivity of the graph.

For an undirected graph:

$$
A_{ij}
=
\begin{cases}
1, & \text{if node } i \text{ is connected to node } j \\
0, & \text{otherwise}
\end{cases}
$$

Because the graph is undirected:

$$
A_{ij}=A_{ji}
$$

At this stage, there are no self-loops, so:

$$
A_{ii}=0
$$

In [ ]:
# Initialize adjacency matrix
A = np.zeros((num_nodes, num_nodes), dtype=np.float32)

# Add both directions because the graph is undirected
for i, j in edges:
    A[i, j] = 1
    A[j, i] = 1

print("Adjacency matrix A:")
print(A)

## 3. Add Self-Loops

In a GCN, each node should also receive its own representation during
message passing.

Therefore, self-loops are added using:

$$
\tilde{A}=A+I
$$

where $I$ is the identity matrix.

The diagonal entries of $\tilde{A}$ are therefore equal to 1.

In [ ]:
# Identity matrix
I = np.eye(num_nodes, dtype=np.float32)

# Add self-loops
A_tilde = A + I

print("Identity matrix I:")
print(I)

print("\nAdjacency matrix with self-loops A_tilde:")
print(A_tilde)

## 4. Construct the Degree Matrix

The degree of node $i$ after adding self-loops is:

$$
\tilde{d}_i
=
\sum_j \tilde{A}_{ij}
$$

For our graph:

$$
\tilde{d}_0=4
$$

$$
\tilde{d}_1=3
$$

$$
\tilde{d}_2=4
$$

$$
\tilde{d}_3=3
$$

$$
\tilde{d}_4=3
$$

Therefore, the degree matrix is:

$$
\tilde{D}
=
\begin{bmatrix}
4&0&0&0&0\\
0&3&0&0&0\\
0&0&4&0&0\\
0&0&0&3&0\\
0&0&0&0&3
\end{bmatrix}
$$

In [ ]:
# Calculate node degrees
degrees = A_tilde.sum(axis=1)

# Construct diagonal degree matrix
D_tilde = np.diag(degrees)

print("Degrees:")
print(degrees)

print("\nDegree matrix D_tilde:")
print(D_tilde)

## 5. Compute the Inverse Square Root of the Degree Matrix

The symmetric normalization used in a Vanilla GCN requires:

$$
\tilde{D}^{-\frac{1}{2}}
$$

Since $\tilde{D}$ is diagonal, each diagonal element can be transformed
independently:

$$
\tilde{D}^{-\frac{1}{2}}_{ii}
=
\frac{1}{\sqrt{\tilde{d}_i}}
$$

Therefore:

$$
\tilde{D}^{-\frac{1}{2}}
=
\begin{bmatrix}
\frac{1}{2}&0&0&0&0\\
0&\frac{1}{\sqrt{3}}&0&0&0\\
0&0&\frac{1}{2}&0&0\\
0&0&0&\frac{1}{\sqrt{3}}&0\\
0&0&0&0&\frac{1}{\sqrt{3}}
\end{bmatrix}
$$

In [ ]:
# Inverse square root of the degree values
D_inv_sqrt_values = 1.0 / np.sqrt(degrees)

# Construct diagonal matrix
D_inv_sqrt = np.diag(D_inv_sqrt_values)

print("D_tilde^(-1/2):")
print(D_inv_sqrt)

## 6. Construct the Normalized Adjacency Matrix

The normalized adjacency matrix used by the Vanilla GCN is:

$$
\boxed{
\hat{A}
=
\tilde{D}^{-\frac{1}{2}}
\tilde{A}
\tilde{D}^{-\frac{1}{2}}
}
$$

Element-wise, this can also be written as:

$$
\hat{A}_{ij}
=
\frac{\tilde{A}_{ij}}
{\sqrt{\tilde{d}_i\tilde{d}_j}}
$$

This normalization ensures that information received from neighboring
nodes is scaled according to their degrees.

In [ ]:
# Symmetric GCN normalization
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

print("Normalized adjacency matrix A_hat:")
print(A_hat)

# Verify symmetry
print("\nIs A_hat symmetric?")
print(np.allclose(A_hat, A_hat.T))

### Verification of the Normalized Matrix

For example, the self-loop contribution for node 0 is:

$$
\hat{A}_{00}
=
\frac{1}{\sqrt{4}\sqrt{4}}
=
\frac{1}{4}
$$

The connection between nodes 0 and 1 is:

$$
\hat{A}_{01}
=
\frac{1}{\sqrt{4}\sqrt{3}}
=
\frac{1}{2\sqrt{3}}
$$

Similarly, the connection between nodes 1 and 2 is:

$$
\hat{A}_{12}
=
\frac{1}{\sqrt{3}\sqrt{4}}
=
\frac{1}{2\sqrt{3}}
$$

Thus, the numerical matrix obtained above directly corresponds to the
analytical normalization used in the mathematical derivation.

## 7. Define the Node Feature Matrix

The initial node representation is:

$$
H^{(0)}=X
$$

Each node has two features.

The feature matrix is:

$$
X
=
\begin{bmatrix}
1&0\\
0&1\\
1&1\\
0&1\\
1&0
\end{bmatrix}
$$

Therefore:

$$
X\in\mathbb{R}^{5\times2}
$$

Each row corresponds to one node:

$$
x_0=[1,0]
$$

$$
x_1=[0,1]
$$

$$
x_2=[1,1]
$$

$$
x_3=[0,1]
$$

$$
x_4=[1,0]
$$

In [ ]:
# Initial node features
X = np.array([
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 1],
    [1, 0]
], dtype=np.float32)

print("Feature matrix X:")
print(X)

print("\nShape of X:", X.shape)

## 8. Define the Node Labels

This is a two-class node classification problem.

The two classes are:

$$
C=\{0,1\}
$$

Only nodes 0 and 3 have known labels during training.

Their labels are:

$$
y_0=0
$$

$$
y_3=1
$$

The remaining nodes are treated as unlabeled during training.

Therefore, the training node set is:

$$
V_L=\{0,3\}
$$

A training mask is used so that the loss is calculated only on these
two labeled nodes.

In [ ]:
# Number of classes
num_classes = 2

# Labels
# The labels of nodes 1, 2 and 4 are placeholders because
# they are not used during training.
y = torch.tensor(
    [0, 0, 0, 1, 0],
    dtype=torch.long
)

# Training mask
train_mask = torch.tensor(
    [True, False, False, True, False]
)

print("Labels:")
print(y)

print("\nTraining mask:")
print(train_mask)

## 9. Convert the Graph Data to PyTorch Tensors

The mathematical matrices $X$ and $\hat{A}$ are converted into PyTorch
tensors.

The graph structure is fixed during training.

The trainable parameters will be the GCN weight matrices:

$$
W^{(0)}
$$

and

$$
W^{(1)}
$$

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float32)
A_hat_tensor = torch.tensor(A_hat, dtype=torch.float32)

print("X tensor shape:", X_tensor.shape)
print("A_hat tensor shape:", A_hat_tensor.shape)

## 10. Define the Vanilla GCN Architecture

We use a two-layer GCN.

The first layer performs:

$$
H^{(1)}
=
ReLU
\left(
\hat{A}H^{(0)}W^{(0)}
\right)
$$

Since:

$$
H^{(0)}=X
$$

the first layer becomes:

$$
H^{(1)}
=
ReLU
\left(
\hat{A}XW^{(0)}
\right)
$$

The second layer produces the class logits:

$$
Z
=
\hat{A}H^{(1)}W^{(1)}
$$

No ReLU is applied after the final layer because the output consists of
logits that will be passed to the softmax operation during prediction and
to the cross-entropy loss during training.

The dimensions are:

$$
X\in\mathbb{R}^{5\times2}
$$

$$
W^{(0)}\in\mathbb{R}^{2\times2}
$$

$$
H^{(1)}\in\mathbb{R}^{5\times2}
$$

$$
W^{(1)}\in\mathbb{R}^{2\times2}
$$

$$
Z\in\mathbb{R}^{5\times2}
$$

In [ ]:
class VanillaGCN(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        # First layer weights
        self.W0 = nn.Parameter(
            torch.empty(input_dim, hidden_dim)
        )

        # Second layer weights
        self.W1 = nn.Parameter(
            torch.empty(hidden_dim, output_dim)
        )

        # Xavier initialization
        nn.init.xavier_uniform_(self.W0)
        nn.init.xavier_uniform_(self.W1)

    def forward(self, X, A_hat):

        # First GCN layer
        H1 = A_hat @ X @ self.W0

        # Non-linearity
        H1 = torch.relu(H1)

        # Second GCN layer
        Z = A_hat @ H1 @ self.W1

        return H1, Z

## 11. Initialize the GCN

The model uses:

- Input dimension = 2
- Hidden dimension = 2
- Output dimension = 2

Therefore:

$$
W^{(0)}\in\mathbb{R}^{2\times2}
$$

and:

$$
W^{(1)}\in\mathbb{R}^{2\times2}
$$

The weights are initially randomized using Xavier initialization.

During training, gradient descent will update these matrices.

In [ ]:
model = VanillaGCN(
    input_dim=2,
    hidden_dim=2,
    output_dim=2
)

print("Initial W^(0):")
print(model.W0.detach().numpy())

print("\nInitial W^(1):")
print(model.W1.detach().numpy())

## 12. Perform the First GCN Layer

The first layer is:

$$
H^{(1)}
=
ReLU
\left(
\hat{A}XW^{(0)}
\right)
$$

The computation occurs in three stages:

$$
XW^{(0)}
$$

followed by graph aggregation:

$$
\hat{A}(XW^{(0)})
$$

and finally the ReLU activation:

$$
H^{(1)}
=
ReLU
\left(
\hat{A}XW^{(0)}
\right)
$$

This allows each node to combine its own features with information from
its graph neighbors.

In [ ]:
# First layer step-by-step

XW0 = X_tensor @ model.W0

print("X @ W^(0):")
print(XW0.detach().numpy())

A_XW0 = A_hat_tensor @ XW0

print("\nA_hat @ X @ W^(0):")
print(A_XW0.detach().numpy())

H1 = torch.relu(A_XW0)

print("\nH^(1) after ReLU:")
print(H1.detach().numpy())

## 13. Perform the Second GCN Layer

The second layer performs another round of graph aggregation:

$$
M
=
\hat{A}H^{(1)}
$$

followed by the trainable transformation:

$$
Z
=
MW^{(1)}
$$

Therefore:

$$
Z
=
\hat{A}H^{(1)}W^{(1)}
$$

The resulting matrix $Z$ contains two logits for every node:

$$
Z\in\mathbb{R}^{5\times2}
$$

Each row corresponds to a node, and each column corresponds to one
of the two classes.

In [ ]:
# Second graph aggregation
M = A_hat_tensor @ H1

print("M = A_hat @ H^(1):")
print(M.detach().numpy())

# Second layer transformation
Z = M @ model.W1

print("\nZ = M @ W^(1):")
print(Z.detach().numpy())

## 14. Convert Logits into Class Probabilities

The logits are converted into probabilities using softmax.

For node $i$ and class $c$:

$$
P_{ic}
=
\frac{e^{Z_{ic}}}
{\sum_{k=1}^{C}e^{Z_{ik}}}
$$

Since there are two classes:

$$
P_{i0}+P_{i1}=1
$$

The predicted class is the class having the highest probability:

$$
\hat{y}_i
=
\arg\max_c P_{ic}
$$

In [ ]:
probabilities = torch.softmax(Z, dim=1)

predictions = torch.argmax(
    probabilities,
    dim=1
)

print("Class probabilities:")
print(probabilities.detach().numpy())

print("\nPredicted classes:")
print(predictions.detach().numpy())

## 15. Cross-Entropy Loss

The model is trained using the labeled nodes only:

$$
V_L=\{0,3\}
$$

For a labeled node $i$, the cross-entropy loss is:

$$
L_i
=
-\log(P_{i,y_i})
$$

The average training loss is:

$$
\mathcal{L}
=
-\frac{1}{|V_L|}
\sum_{i\in V_L}
\log(P_{i,y_i})
$$

Thus, predictions for nodes 1, 2 and 4 do not contribute to the training
loss.

In [ ]:
criterion = nn.CrossEntropyLoss()

loss = criterion(
    Z[train_mask],
    y[train_mask]
)

print("Training loss:", loss.item())

## 16. Backpropagation

The GCN parameters are:

$$
\Theta
=
\left\{
W^{(0)},W^{(1)}
\right\}
$$

The objective is to minimize:

$$
\mathcal{L}
$$

using gradient descent.

The gradients required are:

$$
\frac{\partial\mathcal{L}}
{\partial W^{(0)}}
$$

and:

$$
\frac{\partial\mathcal{L}}
{\partial W^{(1)}}
$$

The gradient descent update rule is:

$$
W^{(k)}_{\text{new}}
=
W^{(k)}_{\text{old}}
-
\eta
\frac{\partial\mathcal{L}}
{\partial W^{(k)}}
$$

where $\eta$ is the learning rate.

PyTorch's automatic differentiation mechanism calculates these derivatives
through the complete computational graph.

In [ ]:
# Clear previous gradients
model.zero_grad()

# Forward pass
H1, Z = model(
    X_tensor,
    A_hat_tensor
)

# Loss on labeled nodes only
loss = criterion(
    Z[train_mask],
    y[train_mask]
)

# Backpropagation
loss.backward()

print("Gradient of W^(0):")
print(model.W0.grad)

print("\nGradient of W^(1):")
print(model.W1.grad)

## 17. Gradient Descent Optimizer

We use stochastic gradient descent with learning rate:

$$
\eta=0.01
$$

At every training step, the weights are updated according to:

$$
W^{(0)}
\leftarrow
W^{(0)}
-
\eta
\frac{\partial\mathcal{L}}
{\partial W^{(0)}}
$$

and:

$$
W^{(1)}
\leftarrow
W^{(1)}
-
\eta
\frac{\partial\mathcal{L}}
{\partial W^{(1)}}
$$

In [ ]:
learning_rate = 0.01

optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate
)

print("Learning rate:", learning_rate)

## 18. Train the Vanilla GCN

The complete training procedure is repeated for multiple epochs.

Each epoch consists of:

1. Forward propagation
2. Calculation of the loss
3. Backpropagation
4. Gradient-based weight update

The process can be summarized as:

$$
X
\rightarrow
H^{(1)}
\rightarrow
Z
\rightarrow
P
\rightarrow
\mathcal{L}
$$

followed by:

$$
\mathcal{L}
\rightarrow
\nabla\mathcal{L}
\rightarrow
W^{(0)},W^{(1)}
$$

The model is trained using only nodes 0 and 3.

In [ ]:
num_epochs = 500

loss_history = []

for epoch in range(num_epochs):

    # Forward propagation
    H1, Z = model(
        X_tensor,
        A_hat_tensor
    )

    # Calculate loss only on labeled nodes
    loss = criterion(
        Z[train_mask],
        y[train_mask]
    )

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Store loss
    loss_history.append(loss.item())

    # Display progress
    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1:3d} | "
            f"Loss: {loss.item():.6f}"
        )

## 19. Training Loss

The loss recorded during training is plotted against the number of epochs.

A decreasing loss indicates that the model is adjusting its trainable
weight matrices to reduce the classification error on the labeled nodes.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, num_epochs + 1),
    loss_history
)

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Vanilla GCN Training Loss")
plt.grid(True)

plt.show()

## 20. Final Node Classification

After training, the final model is used to calculate the logits:

$$
Z
=
\hat{A}H^{(1)}W^{(1)}
$$

The logits are converted into probabilities:

$$
P
=
softmax(Z)
$$

and the predicted class is:

$$
\hat{y}_i
=
\arg\max_c P_{ic}
$$

The model produces predictions for all five nodes, even though only
nodes 0 and 3 were used as labeled training examples.

In [ ]:
model.eval()

with torch.no_grad():

    H1_final, Z_final = model(
        X_tensor,
        A_hat_tensor
    )

    probabilities = torch.softmax(
        Z_final,
        dim=1
    )

    predictions = torch.argmax(
        probabilities,
        dim=1
    )

print("Final logits:")
print(Z_final.numpy())

print("\nFinal class probabilities:")
print(probabilities.numpy())

print("\nFinal predicted classes:")
print(predictions.numpy())

## 21. Training Accuracy

The training accuracy is calculated only on the labeled nodes:

$$
V_L=\{0,3\}
$$

The accuracy is:

$$
Accuracy
=
\frac{
\text{Number of correctly classified labeled nodes}
}{
\text{Number of labeled nodes}
}
$$

In [ ]:
train_predictions = predictions[train_mask]
train_labels = y[train_mask]

train_accuracy = (
    train_predictions == train_labels
).float().mean()

print("Training predictions:", train_predictions.numpy())
print("Training labels:", train_labels.numpy())
print("Training accuracy:", train_accuracy.item())

## 22. Verify the Mathematical GCN Formulation

The implementation should exactly reproduce the mathematical model.

The first layer is:

$$
H^{(1)}
=
ReLU
\left(
\hat{A}XW^{(0)}
\right)
$$

The second layer is:

$$
Z
=
\hat{A}H^{(1)}W^{(1)}
$$

We explicitly reconstruct both expressions using matrix multiplication
and compare them with the output produced by the GCN model.

In [ ]:
model.eval()

with torch.no_grad():

    # Explicit first-layer calculation
    H1_check = torch.relu(
        A_hat_tensor
        @ X_tensor
        @ model.W0
    )

    # Explicit second-layer calculation
    Z_check = (
        A_hat_tensor
        @ H1_check
        @ model.W1
    )

    # Maximum numerical differences
    h1_difference = torch.max(
        torch.abs(H1_check - H1_final)
    ).item()

    z_difference = torch.max(
        torch.abs(Z_check - Z_final)
    ).item()

print("Maximum difference in H^(1):", h1_difference)
print("Maximum difference in Z:", z_difference)

# 23. Implementation Summary

The Vanilla GCN was implemented directly from its mathematical formulation.

The complete process was:

$$
A
\rightarrow
\tilde{A}=A+I
$$

$$
\tilde{D}
\rightarrow
\tilde{D}^{-\frac12}
$$

$$
\hat{A}
=
\tilde{D}^{-\frac12}
\tilde{A}
\tilde{D}^{-\frac12}
$$

The first GCN layer was:

$$
H^{(1)}
=
ReLU
\left(
\hat{A}XW^{(0)}
\right)
$$

The second GCN layer was:

$$
Z
=
\hat{A}H^{(1)}W^{(1)}
$$

The class probabilities were obtained using:

$$
P
=
softmax(Z)
$$

The model was trained using cross-entropy loss on the labeled nodes:

$$
\mathcal{L}
=
-\frac{1}{|V_L|}
\sum_{i\in V_L}
\log(P_{i,y_i})
$$

The parameters $W^{(0)}$ and $W^{(1)}$ were optimized using gradient
descent.

Thus, the implementation demonstrates the complete Vanilla GCN pipeline:

$$
\boxed{
\text{Graph Construction}
\rightarrow
\text{Normalization}
\rightarrow
\text{Feature Propagation}
\rightarrow
\text{Classification}
\rightarrow
\text{Loss}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Training}
}
$$

# Extension: From the Two-Layer Mathematical Model to a Three-Layer GCN

The preceding section demonstrated the complete mathematical operation of a
two-layer Vanilla GCN using a small graph.

The two-layer model was:

$$
H^{(0)}=X
$$

$$
H^{(1)}
=
ReLU
\left(
\hat{A}H^{(0)}W^{(0)}
\right)
$$

$$
Z
=
\hat{A}H^{(1)}W^{(1)}
$$

This small graph was used to explicitly calculate the matrices and intermediate
representations by hand.

For the implementation, we now extend the same Vanilla GCN formulation to
three graph convolutional layers and apply it to a real-world graph dataset.

The purpose of this extension is to demonstrate that the mathematical model
derived above can be implemented and trained on a larger graph.

# Part II: Three-Layer Vanilla GCN on a Real-World Graph

The previous section demonstrated the mathematical formulation of a two-layer
Vanilla GCN using a small graph.

The small graph allowed the complete matrix operations to be examined
numerically by hand.

We now extend the same mathematical formulation to a three-layer GCN and
apply it to the Cora citation network.

The Cora dataset is a real-world citation graph in which:

- Nodes represent scientific papers.
- Edges represent citation relationships.
- Node features represent word-based features of the papers.
- Each node belongs to one of seven research-topic classes.

The graph contains 2,708 nodes, 10,556 edge entries, 1,433 node features,
and 7 classes.

The dataset provides predefined training, validation, and test masks.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

from torch_geometric.datasets import Planetoid

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

## 1. Load the Cora Dataset

The Cora citation network is loaded using the PyTorch Geometric
`Planetoid` dataset loader.

The dataset contains a single graph:

$$
G=(V,E)
$$

Each node has an associated feature vector:

$$
x_i\in\mathbb{R}^{1433}
$$

and a class label:

$$
y_i\in\{0,1,\ldots,6\}
$$

The task is semi-supervised node classification because only a subset of
the nodes are used for training.

In [ ]:
dataset = Planetoid(
    root="data/Cora",
    name="Cora"
)

data = dataset[0]

print(dataset)
print(data)

## 2. Analyze the Dataset

We inspect the dimensions of the graph before constructing the GCN.

The important quantities are:

$$
|V| = \text{number of nodes}
$$

$$
|E| = \text{number of edges}
$$

$$
F = \text{number of node features}
$$

$$
C = \text{number of classes}
$$

The feature matrix therefore has the form:

$$
X\in\mathbb{R}^{|V|\times F}
$$

and the final GCN output has the form:

$$
Z\in\mathbb{R}^{|V|\times C}
$$

In [ ]:
print("Number of nodes:", data.num_nodes)
print("Number of edge entries:", data.num_edges)
print("Number of node features:", dataset.num_node_features)
print("Number of classes:", dataset.num_classes)

print("\nFeature matrix shape:", data.x.shape)
print("Label vector shape:", data.y.shape)

print("\nTraining nodes:", int(data.train_mask.sum()))
print("Validation nodes:", int(data.val_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

print("\nIs graph undirected?", data.is_undirected())

## 3. Construct the Adjacency Matrix

The Cora dataset provides the graph using an edge-index representation.

For the Vanilla GCN formulation, we require the adjacency matrix:

$$
A\in\mathbb{R}^{N\times N}
$$

where:

$$
A_{ij}
=
\begin{cases}
1, & \text{if an edge exists between nodes }i\text{ and }j\\
0, & \text{otherwise}
\end{cases}
$$

Because the Cora graph is undirected, the adjacency matrix is symmetric.

Self-loops are then added:

$$
\tilde A=A+I
$$

In [ ]:
N = data.num_nodes

# Construct dense adjacency matrix for explicit mathematical implementation
A = torch.zeros((N, N), dtype=torch.float32)

src = data.edge_index[0]
dst = data.edge_index[1]

A[src, dst] = 1.0

# Make symmetry explicit
A = torch.maximum(A, A.T)

# Add self-loops
I = torch.eye(N)

A_tilde = A + I

print("A shape:", A.shape)
print("A_tilde shape:", A_tilde.shape)
print("Number of non-zero entries in A:", int((A != 0).sum()))
print("Number of self-loops:", int(torch.diag(A_tilde).sum()))

## 4. Construct the Normalized Adjacency Matrix

The degree of node $i$ is:

$$
\tilde d_i
=
\sum_j \tilde A_{ij}
$$

The degree matrix is:

$$
\tilde D
=
diag(\tilde d_1,\tilde d_2,\ldots,\tilde d_N)
$$

The Vanilla GCN uses symmetric normalization:

$$
\boxed{
\hat A
=
\tilde D^{-\frac12}
\tilde A
\tilde D^{-\frac12}
}
$$

For large real-world graphs, explicitly storing a dense $N\times N$ matrix is
memory-intensive. Therefore, the implementation below constructs the
normalization in sparse form.

The mathematical operation remains exactly:

$$
\hat A
=
\tilde D^{-\frac12}
\tilde A
\tilde D^{-\frac12}
$$

In [ ]:
# Degree of every node after adding self-loops
degrees = A_tilde.sum(dim=1)

# D^(-1/2)
D_inv_sqrt = degrees.pow(-0.5)

# Construct normalized adjacency values
edge_index = A_tilde.nonzero(as_tuple=False).T

row = edge_index[0]
col = edge_index[1]

edge_weight = D_inv_sqrt[row] * D_inv_sqrt[col]

A_hat = torch.sparse_coo_tensor(
    edge_index,
    edge_weight,
    size=(N, N)
).coalesce()

print("Normalized adjacency shape:", A_hat.shape)
print("Number of normalized non-zero entries:",
      A_hat._nnz())

## 5. Three-Layer Vanilla GCN

The initial node representation is:

$$
H^{(0)}=X
$$

The first GCN layer is:

$$
H^{(1)}
=
ReLU
\left(
\hat A H^{(0)}W^{(0)}
\right)
$$

The second GCN layer is:

$$
H^{(2)}
=
ReLU
\left(
\hat A H^{(1)}W^{(1)}
\right)
$$

The third GCN layer produces the output logits:

$$
Z
=
\hat A H^{(2)}W^{(2)}
$$

Therefore:

$$
\boxed{
H^{(0)}
\rightarrow
H^{(1)}
\rightarrow
H^{(2)}
\rightarrow
Z
}
$$

The dimensions are:

$$
W^{(0)}
\in
\mathbb{R}^{1433\times64}
$$

$$
W^{(1)}
\in
\mathbb{R}^{64\times64}
$$

$$
W^{(2)}
\in
\mathbb{R}^{64\times7}
$$

Therefore:

$$
Z\in\mathbb{R}^{2708\times7}
$$

In [ ]:
class ThreeLayerGCN(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.W0 = nn.Parameter(
            torch.empty(input_dim, hidden_dim)
        )

        self.W1 = nn.Parameter(
            torch.empty(hidden_dim, hidden_dim)
        )

        self.W2 = nn.Parameter(
            torch.empty(hidden_dim, output_dim)
        )

        nn.init.xavier_uniform_(self.W0)
        nn.init.xavier_uniform_(self.W1)
        nn.init.xavier_uniform_(self.W2)

    def propagate(self, A_hat, H, W):
        return torch.sparse.mm(A_hat, H) @ W

    def forward(self, X, A_hat):

        H1 = self.propagate(
            A_hat, X, self.W0
        )
        H1 = torch.relu(H1)

        H2 = self.propagate(
            A_hat, H1, self.W1
        )
        H2 = torch.relu(H2)

        Z = self.propagate(
            A_hat, H2, self.W2
        )

        return H1, H2, Z

In [ ]:
X = data.x.float()
y = data.y.long()

model = ThreeLayerGCN(
    input_dim=dataset.num_node_features,
    hidden_dim=64,
    output_dim=dataset.num_classes
)

print("Input dimension:", dataset.num_node_features)
print("Hidden dimension:", 64)
print("Output dimension:", dataset.num_classes)

## 6. Forward Propagation

The forward pass follows the mathematical model directly.

First:

$$
S^{(1)}
=
\hat A XW^{(0)}
$$

then:

$$
H^{(1)}
=
ReLU(S^{(1)})
$$

Second:

$$
S^{(2)}
=
\hat A H^{(1)}W^{(1)}
$$

and:

$$
H^{(2)}
=
ReLU(S^{(2)})
$$

Finally:

$$
Z
=
\hat A H^{(2)}W^{(2)}
$$

The output contains seven logits for every node because Cora contains
seven node classes.

In [ ]:
with torch.no_grad():

    H1, H2, Z = model(
        X,
        A_hat
    )

print("H^(1) shape:", H1.shape)
print("H^(2) shape:", H2.shape)
print("Z shape:", Z.shape)

## 7. Cross-Entropy Loss

The model is trained only using the nodes selected by the training mask.

The training objective is:

$$
\mathcal L
=
-\frac{1}{|V_{train}|}
\sum_{i\in V_{train}}
\log
P_{i,y_i}
$$

where:

$$
P=softmax(Z)
$$

The validation and test nodes do not contribute to the training loss.

This preserves the semi-supervised node-classification setting of the dataset.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

with torch.no_grad():
    _, _, initial_Z = model(X, A_hat)

initial_loss = criterion(
    initial_Z[data.train_mask],
    y[data.train_mask]
)

print("Initial training loss:", initial_loss.item())

## 8. Training the Three-Layer GCN

The trainable parameters are:

$$
\Theta
=
\{
W^{(0)},W^{(1)},W^{(2)}
\}
$$

At every epoch:

1. Perform forward propagation.
2. Calculate the training loss.
3. Compute gradients using backpropagation.
4. Update the three weight matrices.

The update rule is:

$$
W^{(k)}
\leftarrow
W^{(k)}
-
\eta
\frac{\partial\mathcal L}
{\partial W^{(k)}}
$$

for:

$$
k\in\{0,1,2\}
$$

In [ ]:
num_epochs = 300

train_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):

    model.train()

    H1, H2, Z = model(
        X,
        A_hat
    )

    loss = criterion(
        Z[data.train_mask],
        y[data.train_mask]
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Training accuracy
    with torch.no_grad():

        train_pred = Z.argmax(dim=1)

        train_acc = (
            train_pred[data.train_mask]
            == y[data.train_mask]
        ).float().mean()

        val_acc = (
            train_pred[data.val_mask]
            == y[data.val_mask]
        ).float().mean()

    train_losses.append(loss.item())
    train_accuracies.append(train_acc.item())
    val_accuracies.append(val_acc.item())

    if (epoch + 1) % 25 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Train Acc: {train_acc.item():.4f} | "
            f"Val Acc: {val_acc.item():.4f}"
        )

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    train_losses
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Three-Layer Vanilla GCN Training Loss")
plt.grid(True)

plt.show()

## 9. Training Analysis

The training curve is used to determine whether optimization is occurring.

The primary condition we examine is:

$$
\mathcal L_{final}
<
\mathcal L_{initial}
$$

A decreasing loss indicates that the trainable parameters are being adjusted
to reduce the classification error on the labeled training nodes.

Validation accuracy is also monitored because training accuracy alone does not
measure generalization to unseen nodes.

# Part III: Test Cases

The implementation is evaluated using several test cases.

The tests verify:

1. Graph dimensions.
2. Feature dimensions.
3. Normalized adjacency dimensions.
4. Forward-pass dimensions.
5. Probability normalization.
6. Loss validity.
7. Training improvement.
8. Final test-set classification accuracy.

Each test is reported as PASS or FAIL.

In [ ]:
def run_test(name, condition):
    status = "PASS" if condition else "FAIL"
    print(f"{status:4s} | {name}")
    return condition


# Test 1: Graph dimensions
test1 = run_test(
    "Graph contains 2708 nodes",
    data.num_nodes == 2708
)

# Test 2: Feature dimensions
test2 = run_test(
    "Feature matrix has shape (2708, 1433)",
    tuple(X.shape) == (2708, 1433)
)

# Test 3: Number of classes
test3 = run_test(
    "Number of classes is 7",
    dataset.num_classes == 7
)

# Test 4: Normalized adjacency dimensions
test4 = run_test(
    "A_hat has shape (2708, 2708)",
    tuple(A_hat.shape) == (2708, 2708)
)

# Test 5: Forward-pass dimensions
test5 = run_test(
    "Final logits have shape (2708, 7)",
    tuple(Z.shape) == (2708, 7)
)

In [ ]:
model.eval()

with torch.no_grad():

    H1_final, H2_final, Z_final = model(
        X,
        A_hat
    )

    probabilities = torch.softmax(
        Z_final,
        dim=1
    )

    predictions = probabilities.argmax(
        dim=1
    )

probability_sums = probabilities.sum(dim=1)

test6 = run_test(
    "Softmax probabilities sum to 1",
    torch.allclose(
        probability_sums,
        torch.ones_like(probability_sums),
        atol=1e-5
    )
)

print(
    "Maximum probability-sum error:",
    torch.max(
        torch.abs(probability_sums - 1)
    ).item()
)

In [ ]:
final_loss = criterion(
    Z_final[data.train_mask],
    y[data.train_mask]
).item()

test7 = run_test(
    "Final training loss is finite",
    np.isfinite(final_loss)
)

test8 = run_test(
    "Training loss decreased",
    final_loss < initial_loss.item()
)

print("\nInitial loss:", initial_loss.item())
print("Final loss:", final_loss)

## 10. Final Test Evaluation

The test nodes were not used to optimize the model parameters.

After training, the predicted class is:

$$
\hat y_i
=
\arg\max_c Z_{ic}
$$

for each test node.

The test accuracy is:

$$
Accuracy_{test}
=
\frac{
\sum_{i\in V_{test}}
\mathbf{1}(\hat y_i=y_i)
}{
|V_{test}|
}
$$

This provides the final evaluation of the trained three-layer Vanilla GCN.

In [ ]:
test_predictions = predictions[data.test_mask]

test_labels = y[data.test_mask]

test_accuracy = (
    test_predictions == test_labels
).float().mean()

print("Test nodes:", int(data.test_mask.sum()))
print("Correct predictions:",
      int((test_predictions == test_labels).sum()))
print("Test accuracy:",
      test_accuracy.item())

test9 = run_test(
    "Test accuracy is a valid value",
    0.0 <= test_accuracy.item() <= 1.0
)

In [ ]:
results = {
    "Initial Training Loss": initial_loss.item(),
    "Final Training Loss": final_loss,
    "Final Training Accuracy": train_accuracies[-1],
    "Final Validation Accuracy": val_accuracies[-1],
    "Test Accuracy": test_accuracy.item()
}

print("\n===== FINAL RESULTS =====")

for metric, value in results.items():
    print(f"{metric:30s}: {value:.4f}")

## 11. Mathematical Verification of the Three-Layer Implementation

The implementation is verified against the mathematical GCN equations.

The first layer must satisfy:

$$
H^{(1)}
=
ReLU
\left(
\hat A X W^{(0)}
\right)
$$

The second layer must satisfy:

$$
H^{(2)}
=
ReLU
\left(
\hat A H^{(1)} W^{(1)}
\right)
$$

The final layer must satisfy:

$$
Z
=
\hat A H^{(2)} W^{(2)}
$$

Therefore, the implementation directly follows the three-layer Vanilla GCN
mathematical formulation rather than using a pre-built GCN convolution layer.

In [24]:
model.eval()

with torch.no_grad():

    # Explicit mathematical reconstruction
    H1_check = torch.relu(
        torch.sparse.mm(A_hat, X) @ model.W0
    )

    H2_check = torch.relu(
        torch.sparse.mm(A_hat, H1_check) @ model.W1
    )

    Z_check = (
        torch.sparse.mm(A_hat, H2_check)
        @ model.W2
    )

    difference_H1 = torch.max(
        torch.abs(H1_check - H1_final)
    ).item()

    difference_H2 = torch.max(
        torch.abs(H2_check - H2_final)
    ).item()

    difference_Z = torch.max(
        torch.abs(Z_check - Z_final)
    ).item()

print("Maximum difference in H^(1):", difference_H1)
print("Maximum difference in H^(2):", difference_H2)
print("Maximum difference in Z:", difference_Z)

test10 = run_test(
    "Mathematical reconstruction matches model output",
    max(
        difference_H1,
        difference_H2,
        difference_Z
    ) < 1e-6
)

NameError: name 'model' is not defined

# 12. Final Analysis and Conclusion

The implementation extended the manually derived two-layer Vanilla GCN into
a three-layer model.

The final architecture was:

$$
X
\rightarrow
H^{(1)}
\rightarrow
H^{(2)}
\rightarrow
Z
$$

where:

$$
H^{(1)}
=
ReLU
\left(
\hat A X W^{(0)}
\right)
$$

$$
H^{(2)}
=
ReLU
\left(
\hat A H^{(1)} W^{(1)}
\right)
$$

$$
Z
=
\hat A H^{(2)} W^{(2)}
$$

The model was trained using cross-entropy loss on the training nodes and
evaluated on separate validation and test nodes.

The test cases verified the graph structure, tensor dimensions, probability
normalization, training behavior, final predictions, and consistency between
the implemented model and its mathematical formulation.

The experiment therefore demonstrates the complete pipeline:

$$
\boxed{
\text{Graph}
\rightarrow
\text{Normalization}
\rightarrow
\text{Three GCN Layers}
\rightarrow
\text{Loss}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Training}
\rightarrow
\text{Testing}
}
$$